In [1]:
import kaleidocell
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from datetime import datetime
os.environ['TZ'] = 'Europe/Berlin'
import time
time.tzset()

def checkpoint(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

In [149]:
OUT_DIR = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/"
os.makedirs(OUT_DIR, exist_ok=True)

SELECTED_MPS = ["MP1", "MP2", "MP7", "MP8", "MP9", "MP11", "MP14", "MP15", "MP19", "MP20"]

INFERNO_ORANGE = "#f57b17"
 
# ── Custom edge colormap: bright pink (low similarity) -> dark burgundy (high similarity)
from matplotlib.colors import LinearSegmentedColormap
EDGE_CMAP = LinearSegmentedColormap.from_list("pink_burgundy", ["#ffb3c6", "#7d0541"])
 
# ── Distinct node color per MP, spanning pink/violet/red-pink/orange/yellow shades
NODE_PALETTE = {
    "MP1":  "#aa05af",  # 
    "MP2":  "#dd38ec",  # 
    "MP7":  "#e76902",  # 
    "MP8":  "#fb8500",  # 
    "MP9":  "#ffb703",  # 
    "MP11": "#d00000",  # 
    "MP14": "#dd4ea1",  # 
    "MP15": "#ff4d6d",  # 
    "MP19": "#f380f7",  # 
    "MP20": "#ffd60a",  # 
}

In [68]:
results_mp = kaleidocell.load("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_3_nmf_all_gsea_c7/results_mp.kc")
checkpoint(f"Loaded: {len(results_mp['mp_dict'])} MPs")


[2026-08-11 14:07:44] Loaded: 30 MPs


In [114]:
mp_dict = results_mp["mp_dict"]

checkpoint("Checking mp_dict structure")
example_mp = list(mp_dict.keys())[0]
print(type(mp_dict[example_mp]), mp_dict[example_mp])

[2026-08-11 14:14:54] Checking mp_dict structure
<class 'pandas.core.series.Series'> SKAP1       0.012087
CD96        0.008915
FYN         0.008811
THEMIS      0.008810
CD247       0.007919
BCL11B      0.007598
CAMK4       0.007317
SYNE2       0.006829
TOX         0.006793
STAT4       0.006767
ETS1        0.006431
TC2N        0.006250
SAMD3       0.006221
ITK         0.006168
RIPOR2      0.006065
PPP1R16B    0.006018
INPP4B      0.005442
SLC38A1     0.005409
PRKCQ       0.005341
GRAP2       0.004852
RASGRP1     0.004824
IKZF3       0.004697
SLFN12L     0.004618
CD2         0.004411
ANK3        0.004410
RORA        0.004268
IL7R        0.004204
OXNAD1      0.003957
CCDC88C     0.003927
SCML4       0.003805
CCL5        0.003668
CD6         0.003666
PYHIN1      0.003630
LCK         0.003612
KLF12       0.003470
MLLT3       0.003433
TSPAN5      0.003336
NCALD       0.003335
CYFIP2      0.003316
DOCK9       0.003291
ZNF831      0.003271
PBX4        0.003173
SPOCK2      0.002978
IL32        

In [115]:
# ── Gene counts per MP, restricted to selected MPs
# call below if the structure differs (e.g. if it's a dict of gene: score).
n_genes = {mp: len(mp_dict[mp]) for mp in SELECTED_MPS if mp in mp_dict}
missing_from_dict = [mp for mp in SELECTED_MPS if mp not in mp_dict]
if missing_from_dict:
    checkpoint(f"WARNING: {missing_from_dict} not found in mp_dict keys — check key naming (e.g. 'MP1' vs 'mp1' vs 1)")
checkpoint(f"n_genes per selected MP: {n_genes}")
 
checkpoint("Loading Jaccard similarity table")
jaccard_df = pd.read_csv(os.path.join(OUT_DIR, "05_9_mp_jaccard_similarity.csv"))
mp_col1, mp_col2 = jaccard_df.columns[0], jaccard_df.columns[1]

print(jaccard_df.shape)
print(jaccard_df.head())
print(jaccard_df.columns.tolist())

[2026-08-11 14:14:54] n_genes per selected MP: {'MP1': 55, 'MP2': 41, 'MP7': 28, 'MP8': 30, 'MP9': 26, 'MP11': 94, 'MP14': 71, 'MP15': 222, 'MP19': 17, 'MP20': 36}
[2026-08-11 14:14:54] Loading Jaccard similarity table
(435, 3)
   MP1  MP2   jaccard
0  MP1  MP2  0.142857
1  MP1  MP3  0.000000
2  MP1  MP4  0.000000
3  MP1  MP5  0.000000
4  MP1  MP6  0.000000
['MP1', 'MP2', 'jaccard']


In [116]:
# ── Restrict to selected MPs only (both endpoints of each edge must be selected)
jaccard_sel = jaccard_df[
    jaccard_df[mp_col1].isin(SELECTED_MPS) & jaccard_df[mp_col2].isin(SELECTED_MPS)
].copy()
checkpoint(f"Filtered Jaccard table to selected MPs: {jaccard_sel.shape[0]} pairs (from {jaccard_df.shape[0]})")

[2026-08-11 14:14:54] Filtered Jaccard table to selected MPs: 45 pairs (from 435)


In [117]:
THRESHOLD = 0.050
checkpoint(f"Building network with threshold={THRESHOLD}")
 
G = nx.Graph()
G.add_nodes_from(SELECTED_MPS)  # ensures all 10 appear even if isolated (no edges above threshold)

for _, row in jaccard_sel.iterrows():
    if row["jaccard"] >= THRESHOLD:
        G.add_edge(row[mp_col1], row[mp_col2], weight=row["jaccard"])
 
checkpoint(f"Network built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
isolated = list(nx.isolates(G))
if isolated:
    checkpoint(f"Isolated nodes (no edges above threshold={THRESHOLD}): {isolated}")

[2026-08-11 14:14:54] Building network with threshold=0.05
[2026-08-11 14:14:54] Network built: 10 nodes, 6 edges
[2026-08-11 14:14:54] Isolated nodes (no edges above threshold=0.05): ['MP11', 'MP19']


In [118]:
# ── Layout: kamada_kawai_layout only makes sense for the connected part of the
# graph — isolated nodes have no meaningful position relative to it, and letting
# the algorithm place them "arbitrarily" is what causes huge empty canvas space.
# Instead: layout the connected subgraph compactly, then arrange isolated nodes
# in a neat ring around it.
checkpoint("Computing layout")
isolated = list(nx.isolates(G))
connected_nodes = [n for n in G.nodes() if n not in isolated]
 
if len(connected_nodes) > 0 and G.subgraph(connected_nodes).number_of_edges() > 0:
    sub = G.subgraph(connected_nodes)
    pos = nx.kamada_kawai_layout(sub, weight="weight")
    # normalize connected layout to fit within a small central radius
    coords = np.array(list(pos.values()))
    center = coords.mean(axis=0)
    max_r = np.max(np.linalg.norm(coords - center, axis=1)) if len(coords) > 1 else 0.3
    scale = 0.5 / max(max_r, 0.1)
    pos = {n: (p - center) * scale for n, p in pos.items()}
else:
    pos = {}
    max_r = 0.3

# place isolated nodes in a ring around the connected cluster
if isolated:
    ring_radius = max(max_r * 1.8, 1.2)  # ensure enough separation from center cluster
    for i, node in enumerate(isolated):
        angle = 2 * np.pi * i / len(isolated)
        pos[node] = ring_radius * np.array([np.cos(angle), np.sin(angle)])
 
if not pos:  # fallback: no edges at all anywhere
    pos = nx.circular_layout(G)

[2026-08-11 14:14:54] Computing layout


In [119]:
# Jaccard Similarity Distribution barplot

# import matplotlib.pyplot as plt
# INFERNO_PINK = "#ed1a72"

# plt.hist(jaccard_df["jaccard"], bins=50, color=INFERNO_PINK)
# plt.xlabel("Jaccard similarity")
# plt.ylabel("Count")
# plt.savefig(os.path.join(OUT_DIR, "05_10_jaccard_distribution.pdf"))
# plt.close()

# print(jaccard_df["jaccard"].describe())

In [120]:
# ── Collision resolution: kamada_kawai/ring placement only spaces node *centers*,
# not accounting for how large the markers are drawn. Push overlapping nodes apart
# based on their actual size so bigger nodes (more genes) get proportionally more
# clearance. SEPARATION_SCALE is a tunable knob — increase if nodes still touch,
# decrease if the layout ends up too spread out.
SEPARATION_SCALE = 0.006  # data-units of radius per sqrt(node_size in pts^2)
 
def resolve_overlaps(pos, sizes, iterations=300, padding=1.15):
    nodes = list(pos.keys())
    coords = np.array([pos[n] for n in nodes])
    radii = np.array([np.sqrt(sizes[n]) * SEPARATION_SCALE for n in nodes])
    for _ in range(iterations):
        moved = False
        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                delta = coords[i] - coords[j]
                dist = np.linalg.norm(delta)
                min_dist = (radii[i] + radii[j]) * padding
                if dist < min_dist:
                    moved = True
                    direction = delta / dist if dist > 1e-6 else np.array([1.0, 0.0])
                    overlap = (min_dist - dist) / 2
                    coords[i] += direction * overlap
                    coords[j] -= direction * overlap
        if not moved:
            break
    return {n: coords[k] for k, n in enumerate(nodes)}

In [121]:
# node sizes need to be known before overlap resolution, so compute them here
# (same logic as the sizing block below — kept in sync)
_gene_counts_tmp = np.array([n_genes.get(mp, 1) for mp in G.nodes()])
if _gene_counts_tmp.max() > _gene_counts_tmp.min():
    _sizes_tmp = 300 + (_gene_counts_tmp - _gene_counts_tmp.min()) / (_gene_counts_tmp.max() - _gene_counts_tmp.min()) * (3500 - 300)
else:
    _sizes_tmp = np.full(len(_gene_counts_tmp), 1900)
size_lookup = dict(zip(G.nodes(), _sizes_tmp))
 
pos = resolve_overlaps(pos, size_lookup)


In [122]:
# # ── Manual position overrides — pull specific nodes closer to (or further
# # from) the center if the automatic ring placement puts them too far out.
# # Values are in the same normalized data-coordinate space as everything else
# # (center cluster sits roughly within radius ~0.5-1.0). Print pos below to
# # see current coordinates before choosing new ones.
checkpoint(f"Current positions before manual overrides: {pos}")
 
MANUAL_POS_OVERRIDE = {
    "MP19": np.array([-1, 0]),
    "MP11": np.array([1, 0]),
}
for node, coord in MANUAL_POS_OVERRIDE.items():
    if node in pos:
        pos[node] = coord
    else:
        checkpoint(f"WARNING: '{node}' in MANUAL_POS_OVERRIDE not found among graph nodes")

[2026-08-11 14:14:54] Current positions before manual overrides: {'MP1': array([0.53506355, 0.01076089]), 'MP2': array([0.40246574, 0.37266626]), 'MP7': array([-0.2226924 , -0.14951989]), 'MP8': array([-0.30744941,  0.41842109]), 'MP9': array([-0.18910201,  0.14323506]), 'MP14': array([ 0.10341567, -0.35062134]), 'MP15': array([-0.37815894, -0.78282547]), 'MP20': array([0.0564578 , 0.33788341]), 'MP11': array([1.89348158, 0.        ]), 'MP19': array([-1.89348158e+00,  2.31884616e-16])}


In [123]:
# ── Node size scaled by gene count
MIN_SIZE, MAX_SIZE = 1000, 7000  # wider range for clearer visual contrast
gene_counts = np.array([n_genes.get(mp, 1) for mp in G.nodes()])
if gene_counts.max() > gene_counts.min():
    scaled_sizes = MIN_SIZE + (gene_counts - gene_counts.min()) / (gene_counts.max() - gene_counts.min()) * (MAX_SIZE - MIN_SIZE)
else:
    scaled_sizes = np.full(len(gene_counts), (MIN_SIZE + MAX_SIZE) / 2)
node_size_map = dict(zip(G.nodes(), scaled_sizes))
 


In [150]:
checkpoint("Plotting network")
fig, ax = plt.subplots(figsize=(10, 7))
 
node_sizes = [node_size_map[n] for n in G.nodes()]
node_colors = [NODE_PALETTE.get(n, "#cccccc") for n in G.nodes()]

# ── Edges encoded by BOTH color and thickness for Jaccard similarity —
# no numeric labels, the colorscale + width convey the value instead.
edge_list = list(G.edges())
edge_weights = np.array([G[u][v]["weight"] for u, v in edge_list])
 

if len(edge_weights) > 0:
    norm = plt.Normalize(vmin=edge_weights.min(), vmax=edge_weights.max())
    edge_colors = [EDGE_CMAP(norm(w)) for w in edge_weights]
    edge_widths = 1.5 + (edge_weights - edge_weights.min()) / max(np.ptp(edge_weights), 1e-9) * 6.5
else:
    edge_colors = []
    edge_widths = []


nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, edgecolors="black", linewidths=1.2, ax=ax)
 
# scale each label's font size to its own node's marker size, so labels on
# small nodes (few genes) don't overflow past the node's edge
MIN_FONT, MAX_FONT = 6, 11
for node in G.nodes():
    size = node_size_map[node]
    size_frac = (size - MIN_SIZE) / max(MAX_SIZE - MIN_SIZE, 1e-9)
    font_size = MIN_FONT + size_frac * (MAX_FONT - MIN_FONT)
    nx.draw_networkx_labels(
        G, pos, labels={node: node}, font_size=font_size, font_weight="bold", ax=ax
    )

nx.draw_networkx_edges(G, pos, edgelist=edge_list, width=edge_widths, edge_color=edge_colors, alpha=0.85, ax=ax)
 
# ── Triangular colorbar: combines the color gradient AND the thickness cue
# into one shape — thin/light at the bottom (low similarity) widening to
# thick/dark at the top (high similarity), matching the edges' encoding.
if len(edge_weights) > 0:
    from matplotlib.patches import Polygon
    from matplotlib.transforms import Bbox
 
    cbar_box = Bbox.from_bounds(0.935, 0.15, 0.0255, 0.55)  # [left, bottom, width, height] in figure fraction
    tri_ax = fig.add_axes([cbar_box.x0, cbar_box.y0, cbar_box.width, cbar_box.height])
 
    gradient = np.linspace(edge_weights.min(), edge_weights.max(), 256).reshape(-1, 1)
    im = tri_ax.imshow(
        gradient, extent=[0, 1, 0, 1], origin="lower", aspect="auto",
        cmap=EDGE_CMAP, vmin=edge_weights.min(), vmax=edge_weights.max(),
    )
 
    max_half_width = 0.5
    apex_half_width = 0.03
    triangle = Polygon(
        [(0.5 - apex_half_width, 0.0), (0.5 - max_half_width, 1.0),
         (0.5 + max_half_width, 1.0), (0.5 + apex_half_width, 0.0)],
        closed=True, transform=tri_ax.transData,
    )
    im.set_clip_path(triangle)
 
    tri_ax.set_xlim(0, 1)
    tri_ax.set_ylim(0, 1)
    tri_ax.axis("off")
 
    # min/max value labels alongside the triangle
    tri_ax.text(0.5 + max_half_width + 0.08, 1.0, f"{edge_weights.max():.2f}", va="top", ha="left", fontsize=9)
    tri_ax.text(0.5 + apex_half_width + 0.08, 0.0, f"{edge_weights.min():.2f}", va="bottom", ha="left", fontsize=9)
    tri_ax.text(0.5, 1.12, "Jaccard\nsimilarity", ha="center", va="bottom", fontsize=9, transform=tri_ax.transAxes)
 
# tighten bounds to the actual node positions instead of leaving huge default margins
coords = np.array(list(pos.values()))
pad = 0.35
ax.set_xlim(coords[:, 0].min() - pad, coords[:, 0].max() + pad)
ax.set_ylim(coords[:, 1].min() - pad, coords[:, 1].max() + pad)
ax.set_aspect("equal")
 
ax.axis("off")
ax.set_title(f"Selected MP Gene-Set Jaccard Similarity Network \n (threshold={THRESHOLD}, node size = number of genes )", y = 0.90)
out_path = os.path.join(OUT_DIR, "05_10_mp_jaccard_network_selected_05.pdf")
plt.savefig(out_path, bbox_inches="tight")
plt.close()
checkpoint(f"Network plot saved to {out_path}")
checkpoint("ALL DONE!")

[2026-08-11 15:18:12] Plotting network
[2026-08-11 15:18:12] Network plot saved to /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_10_mp_jaccard_network_selected_05.pdf
[2026-08-11 15:18:12] ALL DONE!
